In [7]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from dataset import MaestroDataset

In [8]:
class PianoTranscriptionNet(nn.Module):
    def __init__(self, input_bins=88, lstm_units=256):
        super(PianoTranscriptionNet, self).__init__()
        
        # 1. Shared CNN Acoustic Layer
        # We use padding to ensure the temporal dimension (Width) stays exactly the same size.
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding=(1, 1))
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=(1, 1))
        self.bn2 = nn.BatchNorm2d(64)
        
        # After flattening the channel and frequency dimensions, calculate the feature size per frame.
        # Channels (64) * Frequency Bins (88) = 5632
        cnn_out_features = 64 * input_bins
        
        # 2. Onset Detection Head
        self.onset_lstm = nn.LSTM(cnn_out_features, lstm_units, batch_first=True, bidirectional=True)
        self.onset_fc = nn.Linear(lstm_units * 2, input_bins) # * 2 because it's bidirectional
        
        # 3. Frame Detection Head
        # The Frame head takes the raw CNN features AND the predicted onset states as input.
        # Input size: cnn_out_features (5632) + predicted onsets (88) = 5720
        self.frame_lstm = nn.LSTM(cnn_out_features + input_bins, lstm_units, batch_first=True, bidirectional=True)
        self.frame_fc = nn.Linear(lstm_units * 2, input_bins)

    def forward(self, x):
        # x shape: [Batch, 1, 88, TimeFrames]
        batch_size, channels, bins, time_frames = x.size()
        
        # Run through CNN
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out))) # Shape: [Batch, 64, 88, TimeFrames]
        
        # Reshape for LSTM sequence processing
        # Target shape: [Batch, TimeFrames, Channels * Bins]
        out = out.permute(0, 3, 1, 2).contiguous()
        out = out.view(batch_size, time_frames, -1)
        
        # --- Head 1: Predict Onsets ---
        onset_out, _ = self.onset_lstm(out)
        onset_logits = self.onset_fc(onset_out) # Shape: [Batch, TimeFrames, 88]
        onset_probs = torch.sigmoid(onset_logits)
        
        # --- Head 2: Predict Frames (Conditioned on Onset Probabilities) ---
        # Concatenate the acoustic features with the predicted onset values along the feature axis
        frame_input = torch.cat([out, onset_probs], dim=-1)
        
        frame_out, _ = self.frame_lstm(frame_input)
        frame_logits = self.frame_fc(frame_out) # Shape: [Batch, TimeFrames, 88]
        frame_probs = torch.sigmoid(frame_logits)
        
        # Rearrange outputs back to match dataset targets: [Batch, 88, TimeFrames]
        return frame_probs.permute(0, 2, 1), onset_probs.permute(0, 2, 1)

In [9]:
class TranscriptionLoss(nn.Module):
    def __init__(self, onset_weight=5.0, frame_weight=1.0):
        super(TranscriptionLoss, self).__init__()
        self.onset_weight = onset_weight
        self.frame_weight = frame_weight

    def forward(self, pred_frames, pred_onsets, target_frames, target_onsets):
        # Using a higher pos_weight handles the data imbalance issue
        # We tell the loss function that missing an actual note is much worse than guess-firing a zero.
        pos_weight_onset = torch.tensor([10.0]).to(pred_onsets.device) # Onsets are extremely sparse
        pos_weight_frame = torch.tensor([3.0]).to(pred_frames.device)
        
        loss_onsets = F.binary_cross_entropy(pred_onsets, target_onsets, weight=pos_weight_onset)
        loss_frames = F.binary_cross_entropy(pred_frames, target_frames, weight=pos_weight_frame)
        
        # Combine the losses using hyperparameters
        total_loss = (self.onset_weight * loss_onsets) + (self.frame_weight * loss_frames)
        return total_loss

In [10]:
torch.set_float32_matmul_precision('high')

# 1. Device Configuration
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Initialize Datasets and Loaders
# Assumes OUTPUT_DIR points to your preprocessed .npz files from earlier steps
PREPROCESSED_DIR = "maestro_preprocessed_complete/"

train_dataset = MaestroDataset(preprocessed_dir=PREPROCESSED_DIR, split='train', sequence_length=128)
val_dataset = MaestroDataset(preprocessed_dir=PREPROCESSED_DIR, split='validation', sequence_length=128)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2, pin_memory=False)

# 3. Instantiate Model, Loss, and Optimizer
model = PianoTranscriptionNet(input_bins=88, lstm_units=256).to(device)
criterion = TranscriptionLoss(onset_weight=5.0, frame_weight=1.0).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
# Automatically scale down the learning rate if validation loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# 4. Training Loop
NUM_EPOCHS = 20
best_val_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    running_train_loss = 0.0
    
    for batch_idx, (batch_x, batch_y_frames, batch_y_onsets) in enumerate(train_loader):
        # Move tensors to GPU
        inputs = batch_x.to(device)
        target_frames = batch_y_frames.to(device)
        target_onsets = batch_y_onsets.to(device)
        
        # Zero out gradients from the previous step
        optimizer.zero_grad()
        
        # Forward pass
        pred_frames, pred_onsets = model(inputs)
        
        # Calculate joint loss
        loss = criterion(pred_frames, pred_onsets, target_frames, target_onsets)
        
        # Backward pass
        loss.backward()
        
        # Gradient Clipping to prevent exploding gradients in LSTMs
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update weights
        optimizer.step()
        
        running_train_loss += loss.item()
        
        # Print progress every 10 batches
        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Batch [{batch_idx}/{len(train_loader)}] | Loss: {loss.item():.4f}")

    epoch_train_loss = running_train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    model.eval()
    running_val_loss = 0.0
    
    # Disable gradient tracking to save memory and compute speed
    with torch.no_grad():
        for batch_x, batch_y_frames, batch_y_onsets in val_loader:
            inputs = batch_x.to(device)
            target_frames = batch_y_frames.to(device)
            target_onsets = batch_y_onsets.to(device)
            
            pred_frames, pred_onsets = model(inputs)
            loss = criterion(pred_frames, pred_onsets, target_frames, target_onsets)
            running_val_loss += loss.item()
            
    epoch_val_loss = running_val_loss / len(val_loader)
    
    # Step the learning rate scheduler based on validation loss
    scheduler.step(epoch_val_loss)
    
    # Manually check the current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Current Learning Rate: {current_lr}")
    
    print(f"\n==== Epoch {epoch+1} Complete ====")
    print(f"Avg Train Loss: {epoch_train_loss:.4f} | Avg Val Loss: {epoch_val_loss:.4f}\n")
    
    # --- SAVE BEST MODEL ---
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), "best_piano_transcriber.pth")
        print("--> Saved new best model checkpoint!")

print("Training finished successfully!")

Using device: mps


Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=98, pipe_handle=136)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=98, pipe_handle=143)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aadityakhurana/.pyenv/versions/3.14.0/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/aadityakhurana/.pyenv/versions/3.14.0/lib/python3.14/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Users/aadityakhurana/.pyenv/versions/3.14.0/lib/python3.14/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent

RuntimeError: DataLoader worker (pid(s) 67301) exited unexpectedly